# Module 8: Distributed Training for Large-Scale Models
## Multi-GPU and Multi-Node Training Strategies

This notebook covers:
1. **Multi-GPU training** (8-64 GPUs)
2. **Distributed Data Parallel (DDP)** for scaling across nodes
3. **Tensor Parallelism** for very large models
4. **Pipeline Parallelism** for deep models
5. **FSDP (Fully Sharded Data Parallel)** for 100B+ models
6. **M3 MacBook considerations** and limitations
7. **Resource monitoring** and optimization

**Expected speedup**: 4-8x on 8 GPUs, 50-100x on 64 GPUs

**Target models**: 70B-1T+ parameters

## 1. Import Required Libraries

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler
import psutil

import numpy as np
import time
import math
from dataclasses import dataclass
from typing import Optional, Tuple

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 2. Check Available GPU Resources

In [ ]:
def check_gpu_resources():
    """
    Comprehensive GPU resource detection and reporting
    Works with CUDA, MPS (M3), and CPU
    """
    print("="*70)
    print("GPU RESOURCE REPORT")
    print("="*70)
    
    # CUDA GPUs
    if torch.cuda.is_available():
        num_gpus = torch.cuda.device_count()
        print(f"\n✓ CUDA GPUs: {num_gpus}")
        
        for i in range(num_gpus):
            props = torch.cuda.get_device_properties(i)
            print(f"\n  GPU {i}: {props.name}")
            print(f"    - Memory: {props.total_memory / 1e9:.1f} GB")
            print(f"    - Compute Capability: {props.major}.{props.minor}")
            print(f"    - Multi-processors: {props.multiprocessor_count}")
            print(f"    - Clock Rate: {props.clock_rate / 1e3:.1f} MHz")
            
            # Memory stats
            reserved = torch.cuda.memory_reserved(i) / 1e9
            allocated = torch.cuda.memory_allocated(i) / 1e9
            print(f"    - Memory (reserved/allocated): {reserved:.1f} / {allocated:.1f} GB")
    else:
        print("\n✗ CUDA GPUs: Not available")
    
    # MPS (Apple Silicon)
    if torch.backends.mps.is_available():
        print(f"\n✓ Apple Silicon MPS: Available")
        # MPS doesn't expose detailed specs in PyTorch
        if torch.backends.mps.is_built():
            print(f"    - MPS built: Yes")
            # Detect M3 variant
            try:
                import platform
                model = platform.processor()
                print(f"    - Processor: {model}")
            except:
                print(f"    - Processor: Apple Silicon (M-series)")
    else:
        print(f"\n✗ Apple Silicon MPS: Not available")
    
    # CPU info
    # CPU info (FIXED VERSION)
    
    print(f"\n✓ CPU Cores: {os.cpu_count()}")
    vm = psutil.virtual_memory()
    print(f"  - Total System RAM: {vm.total / 1e9:.1f} GB")

    print("\n" + "="*70)

# Run check
check_gpu_resources()

## 3. Select Optimal Device and Strategy

In [ ]:
def select_device_and_strategy():
    """
    Automatically select best device and training strategy
    """
    
    strategy_info = {
        'device': None,
        'num_gpus': 0,
        'strategy': None,
        'world_size': 1,
        'rank': 0,
    }
    
    # Priority: CUDA multi-GPU > CUDA single > MPS > CPU
    if torch.cuda.is_available():
        num_gpus = torch.cuda.device_count()
        strategy_info['device'] = 'cuda'
        strategy_info['num_gpus'] = num_gpus
        
        if num_gpus >= 4:
            strategy_info['strategy'] = 'fsdp'  # Fully Sharded Data Parallel
            print(f"Strategy: FSDP (Fully Sharded Data Parallel) on {num_gpus} GPUs")
        elif num_gpus >= 2:
            strategy_info['strategy'] = 'ddp'  # Distributed Data Parallel
            print(f"Strategy: DDP (Distributed Data Parallel) on {num_gpus} GPUs")
        else:
            strategy_info['strategy'] = 'dp'  # Data Parallel (single GPU)
            print(f"Strategy: DataParallel (single GPU optimization)")
    
    elif torch.backends.mps.is_available():
        strategy_info['device'] = 'mps'
        strategy_info['num_gpus'] = 1  # M3 can't do multi-GPU easily
        strategy_info['strategy'] = 'mps_single'
        print(f"Strategy: MPS (Apple Silicon) with single GPU optimization")
    
    else:
        strategy_info['device'] = 'cpu'
        strategy_info['num_gpus'] = 0
        strategy_info['strategy'] = 'cpu'
        print(f"Strategy: CPU-only training")
    
    return strategy_info

strategy = select_device_and_strategy()
device = torch.device(strategy['device'])
print(f"\nSelected device: {device}")

## 4. Distributed Training Setup (DDP)

In [ ]:
class DistributedTrainingSetup:
    """
    Setup for Distributed Data Parallel training across multiple GPUs
    
    Usage:
    - Single machine, 8 GPUs:
      torchrun --nproc_per_node=8 train.py
    
    - Multi-node, 2 nodes × 8 GPUs:
      torchrun --nproc_per_node=8 --nnodes=2 \
               --node_rank=0 --master_addr=10.0.0.1 \
               --master_port=29500 train.py
    """
    
    def __init__(self, backend='nccl'):
        self.backend = backend  # 'nccl' for GPU, 'gloo' for CPU
        self.initialized = False
    
    def initialize(self):
        """
        Initialize distributed training.
        Called automatically by torchrun.
        """
        
        # Get rank from environment (set by torchrun)
        if 'RANK' in os.environ:
            rank = int(os.environ['RANK'])
            world_size = int(os.environ['WORLD_SIZE'])
            local_rank = int(os.environ['LOCAL_RANK'])
            
            print(f"Distributed training setup:")
            print(f"  - Global rank: {rank}/{world_size}")
            print(f"  - Local rank: {local_rank}")
            
            # Set GPU for this process
            if torch.cuda.is_available():
                torch.cuda.set_device(local_rank)
            
            # Initialize process group
            dist.init_process_group(backend=self.backend)
            self.initialized = True
        else:
            print("Not running under torchrun. Skipping distributed init.")
            self.initialized = False
        
        return self.initialized
    
    def get_sampler(self, dataset, shuffle=True):
        """
        Create a distributed sampler for the dataset.
        Ensures each GPU gets different data.
        """
        if self.initialized:
            return torch.utils.data.distributed.DistributedSampler(
                dataset,
                shuffle=shuffle,
                drop_last=True,  # Drop incomplete final batch
            )
        else:
            return torch.utils.data.RandomSampler(dataset)
    
    def get_world_size(self):
        return dist.get_world_size() if self.initialized else 1
    
    def get_rank(self):
        return dist.get_rank() if self.initialized else 0

# Demo setup (not initialized without torchrun)
ddp_setup = DistributedTrainingSetup(backend='nccl')
print(f"World size: {ddp_setup.get_world_size()}")
print(f"Rank: {ddp_setup.get_rank()}")

## 5. Fully Sharded Data Parallel (FSDP) - For 100B+ Models

In [ ]:
class OptimizedTransformer(nn.Module):
    """Transformer model suitable for FSDP training"""
    
    def __init__(self, config):
        super().__init__()
        self.embed = nn.Embedding(config.vocab_size, config.hidden_dim)
        
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=config.hidden_dim,
                nhead=config.num_heads,
                dim_feedforward=config.ffn_dim,
                dropout=config.dropout,
                batch_first=True,
            ),
            num_layers=config.num_layers,
        )
        
        self.output = nn.Linear(config.hidden_dim, config.vocab_size)
    
    def forward(self, x):
        x = self.embed(x)
        x = self.transformer(x)
        x = self.output(x)
        return x

@dataclass
class FSDPConfig:
    """Configuration for FSDP training"""
    # Model
    hidden_dim: int = 768
    num_heads: int = 12
    num_layers: int = 24
    ffn_dim: int = 3072
    vocab_size: int = 50257
    dropout: float = 0.1
    
    # Training
    batch_size: int = 32
    learning_rate: float = 1e-4
    num_epochs: int = 3
    gradient_accumulation_steps: int = 4
    
    # FSDP
    use_fsdp: bool = True
    sharding_strategy: str = 'FULL_SHARD'  # or 'SHARD_GRAD_OP', 'NO_SHARD'
    cpu_offload: bool = False  # Set True for 100B+ models
    backward_prefetch: bool = True
    
    # Optimization
    use_gradient_checkpointing: bool = True
    use_mixed_precision: bool = True
    max_norm: float = 1.0

def setup_fsdp_model(model, config):
    """
    Wrap model with FSDP for distributed training
    
    Memory efficiency:
    - Full precision: 175B params × 4 bytes = 700 GB
    - With FSDP on 8 GPUs: 700 GB / 8 = 87.5 GB per GPU
    - With mixed precision: 87.5 GB / 2 = 43.75 GB per GPU
    - With gradient checkpointing: Further 50% memory savings
    """
    
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        print("FSDP requires multiple GPUs. Skipping FSDP setup.")
        return model
    
    if config.use_fsdp:
        # Auto-wrap policy: wrap layers with > 100M parameters
        auto_wrap_policy = size_based_auto_wrap_policy(
            min_num_params=1e8,
        )
        
        model = FSDP(
            model,
            auto_wrap_policy=auto_wrap_policy,
            sharding_strategy=config.sharding_strategy,
            cpu_offload=config.cpu_offload,
            backward_prefetch=config.backward_prefetch,
            device_id=torch.cuda.current_device(),
        )
        
        print(f"✓ FSDP enabled with strategy: {config.sharding_strategy}")
    
    return model

# Example FSDP setup (won't run without distributed environment)
config = FSDPConfig(
    hidden_dim=768,
    num_layers=12,  # Smaller for demo
)
print(f"FSDP config: {config.num_layers} layers × {config.hidden_dim} dim")

## 6. Multi-GPU Training with DataParallel

In [ ]:
class MultiGPUTrainer:
    """
    Simple multi-GPU training using DataParallel
    (Use DDP/FSDP for production)
    """
    
    def __init__(self, model, device='cuda', use_dp=True):
        self.device = device
        
        # Multi-GPU setup
        if use_dp and torch.cuda.device_count() > 1:
            print(f"Using DataParallel with {torch.cuda.device_count()} GPUs")
            model = nn.DataParallel(model)
        
        self.model = model.to(device)
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4)
        self.scaler = GradScaler(device=device)
    
    def train_step(self, batch, use_amp=True, gradient_accumulation_steps=1):
        """
        Single training step with gradient accumulation
        """
        input_ids, labels = batch
        input_ids = input_ids.to(self.device)
        labels = labels.to(self.device)
        
        if use_amp:
            with autocast(device_type='cuda', dtype=torch.float16):
                logits = self.model(input_ids)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1))
        else:
            logits = self.model(input_ids)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1))
        
        # Scale loss for gradient accumulation
        loss = loss / gradient_accumulation_steps
        
        if use_amp:
            self.scaler.scale(loss).backward()
        else:
            loss.backward()
        
        return loss.item() * gradient_accumulation_steps
    
    def optimizer_step(self, use_amp=True):
        """Optimizer step with gradient clipping"""
        
        if use_amp:
            self.scaler.unscale_(self.optimizer)
        
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        
        if use_amp:
            self.scaler.step(self.optimizer)
            self.scaler.update()
        else:
            self.optimizer.step()
        
        self.optimizer.zero_grad()

# Demo
if torch.cuda.device_count() >= 1:
    model = OptimizedTransformer(FSDPConfig())
    trainer = MultiGPUTrainer(model, device='cuda' if torch.cuda.is_available() else 'cpu')
    print(f"✓ Trainer initialized with {torch.cuda.device_count()} GPU(s)")
else:
    print("No CUDA GPUs available for demo")

## 7. Memory Management for Large Models

In [ ]:
def calculate_model_memory(num_params, batch_size, seq_length, use_mixed_precision=True):
    """
    Estimate memory requirements for training
    
    Memory breakdown:
    - Model weights: num_params × dtype_bytes
    - Optimizer states: num_params × 2 × 4 (AdamW: momentum + variance)
    - Activations: batch_size × seq_length × hidden_dim × num_layers × dtype_bytes
    - Gradients: same as weights
    """
    
    # Model weights
    dtype_bytes = 2 if use_mixed_precision else 4  # FP16 or FP32
    weights_memory = num_params * dtype_bytes / 1e9  # GB
    
    # Optimizer states (AdamW always FP32)
    optimizer_memory = num_params * 2 * 4 / 1e9  # GB
    
    # Gradients (same as weights)
    gradient_memory = weights_memory
    
    # Activations (rough estimate)
    # Transformer: 4 × batch_size × seq_len × hidden_dim × num_layers / 1e9
    # (rough multiplier of 4 for various layers)
    activation_memory = 4 * batch_size * seq_length * 768 * 12 * dtype_bytes / 1e9  # Placeholder for 768d, 12L
    
    # Total
    total_memory = weights_memory + optimizer_memory + gradient_memory + activation_memory
    
    return {
        'weights_gb': weights_memory,
        'optimizer_gb': optimizer_memory,
        'gradients_gb': gradient_memory,
        'activations_gb': activation_memory,
        'total_gb': total_memory,
    }

def estimate_training_requirements():
    """Show requirements for common model sizes"""
    
    print("\n" + "="*70)
    print("MEMORY REQUIREMENTS FOR DIFFERENT MODEL SIZES")
    print("="*70)
    
    models = [
        ('LLaMA-7B', 7e9),
        ('LLaMA-13B', 13e9),
        ('LLaMA-70B', 70e9),
        ('GPT-3 (175B)', 175e9),
    ]
    
    configs = [
        ('M3 Pro (18GB)', 4, 2048, True),
        ('Single A100 (80GB)', 32, 2048, True),
        ('8x A100 (80GB each)', 32, 2048, True),
    ]
    
    for model_name, num_params in models:
        print(f"\n{model_name} ({num_params/1e9:.0f}B parameters):")
        
        for config_name, batch_size, seq_len, use_amp in configs:
            memory = calculate_model_memory(num_params, batch_size, seq_len, use_amp)
            total_per_gpu = memory['total_gb']
            
            # For multi-GPU, estimate per GPU
            if '8x' in config_name:
                per_gpu = total_per_gpu / 8
                print(f"  {config_name}: {per_gpu:.1f} GB per GPU")
            else:
                print(f"  {config_name}: {total_per_gpu:.1f} GB total")
    
    print("\n" + "="*70)

estimate_training_requirements()

## 8. M3 MacBook Specific Configuration

In [ ]:
class M3Optimizer:
    """
    Optimizations specific to Apple Silicon M3
    
    Limitations:
    - Single GPU (can't connect multiple M3 Macs easily)
    - Limited unified memory (8-128 GB shared)
    - No tensor parallelism
    - Some PyTorch ops not optimized for MPS
    
    Advantages:
    - Excellent power efficiency (1-2 TFLOPS per watt)
    - Zero data copy overhead (unified memory)
    - Great for development and fine-tuning
    """
    
    @staticmethod
    def get_optimal_config(model_size='small'):
        """
        Get optimal training config for M3 MacBook
        """
        
        configs = {
            'small': {  # For M3 Pro 18GB
                'batch_size': 4,
                'seq_length': 1024,
                'hidden_dim': 256,
                'num_layers': 6,
                'num_heads': 4,
                'gradient_checkpointing': True,
                'mixed_precision': True,
                'max_params': 500e6,  # 500M parameters
            },
            'medium': {  # For M3 Max 36GB
                'batch_size': 8,
                'seq_length': 2048,
                'hidden_dim': 768,
                'num_layers': 12,
                'num_heads': 12,
                'gradient_checkpointing': True,
                'mixed_precision': True,
                'max_params': 7e9,  # 7B parameters
            },
            'large': {  # For M3 Max 128GB
                'batch_size': 16,
                'seq_length': 4096,
                'hidden_dim': 1024,
                'num_layers': 24,
                'num_heads': 16,
                'gradient_checkpointing': True,
                'mixed_precision': True,
                'max_params': 13e9,  # 13B parameters
            }
        }
        
        return configs.get(model_size, configs['small'])
    
    @staticmethod
    def setup_for_m3():
        """
        Configure PyTorch optimally for M3 MacBook
        """
        
        config = {
            # Device settings
            'device': 'mps' if torch.backends.mps.is_available() else 'cpu',
            
            # Critical for M3: No multiprocessing!
            'num_workers': 0,  # ⚠️ MUST be 0 on M3!
            'pin_memory': False,  # Doesn't apply to MPS
            
            # Optimization flags
            'mixed_precision': True,  # FP16, crucial on M3
            'gradient_checkpointing': True,  # Save memory
            'cpu_offload_optimizer': True,  # Move optimizer to CPU RAM
            
            # Batch settings
            'batch_size': 8,
            'gradient_accumulation_steps': 1,  # Increase if OOM
            'max_grad_norm': 1.0,
            
            # Learning rate
            'learning_rate': 1e-4,
            'warmup_steps': 1000,
            
            # Monitoring
            'log_interval': 100,
            'eval_interval': 500,
        }
        
        return config

# Demo M3 configuration
m3_config = M3Optimizer.setup_for_m3()
print("M3 MacBook Optimal Configuration:")
for key, value in m3_config.items():
    print(f"  {key}: {value}")

## 9. Resource Monitoring and Profiling

In [ ]:
class ResourceMonitor:
    """
    Monitor GPU/CPU/Memory utilization during training
    """
    
    def __init__(self, device='cuda'):
        self.device = device
        self.metrics = {
            'gpu_memory_allocated': [],
            'gpu_memory_reserved': [],
            'step_time': [],
            'throughput': [],
        }
    
    def start_profiling(self):
        """Start a profiling session"""
        if 'cuda' in str(self.device):
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        elif 'mps' in str(self.device):
            torch.mps.synchronize()
        
        self.start_time = time.time()
    
    def end_profiling(self, batch_size: int, seq_length: int):
        """End profiling and record metrics"""
        if 'cuda' in str(self.device):
            torch.cuda.synchronize()
        elif 'mps' in str(self.device):
            torch.mps.synchronize()
        
        elapsed = time.time() - self.start_time
        
        # Record metrics
        if 'cuda' in str(self.device):
            allocated = torch.cuda.memory_allocated(self.device) / 1e9
            reserved = torch.cuda.memory_reserved(self.device) / 1e9
            self.metrics['gpu_memory_allocated'].append(allocated)
            self.metrics['gpu_memory_reserved'].append(reserved)
        
        # Throughput (tokens per second)
        tokens = batch_size * seq_length
        throughput = tokens / elapsed
        
        self.metrics['step_time'].append(elapsed)
        self.metrics['throughput'].append(throughput)
        
        return elapsed, throughput
    
    def print_summary(self):
        """Print profiling summary"""
        
        if not self.metrics['step_time']:
            print("No profiling data collected")
            return
        
        print("\n" + "="*70)
        print("PROFILING SUMMARY")
        print("="*70)
        
        # Time metrics
        step_times = self.metrics['step_time']
        avg_time = np.mean(step_times)
        std_time = np.std(step_times)
        
        print(f"\nStep Time (ms):")
        print(f"  - Mean: {avg_time*1000:.2f}")
        print(f"  - Std: {std_time*1000:.2f}")
        print(f"  - Min: {min(step_times)*1000:.2f}")
        print(f"  - Max: {max(step_times)*1000:.2f}")
        
        # Throughput
        throughputs = self.metrics['throughput']
        avg_throughput = np.mean(throughputs)
        
        print(f"\nThroughput (tokens/sec):")
        print(f"  - Mean: {avg_throughput:.1f}")
        print(f"  - Min: {min(throughputs):.1f}")
        print(f"  - Max: {max(throughputs):.1f}")
        
        # GPU Memory
        if self.metrics['gpu_memory_allocated']:
            allocated = self.metrics['gpu_memory_allocated']
            print(f"\nGPU Memory Allocated (GB):")
            print(f"  - Mean: {np.mean(allocated):.1f}")
            print(f"  - Peak: {max(allocated):.1f}")
        
        print("\n" + "="*70)

# Demo
monitor = ResourceMonitor(device='cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Resource monitor initialized for {monitor.device}")

## 10. Complete Training Loop with Optimization

In [ ]:
def run_optimized_training_example():
    """
    Complete example showing all optimizations together
    """
    
    print("\n" + "="*70)
    print("OPTIMIZED MULTI-GPU TRAINING EXAMPLE")
    print("="*70)
    
    # Select device
    if torch.cuda.is_available():
        device = torch.device('cuda')
        num_gpus = torch.cuda.device_count()
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
        num_gpus = 1
    else:
        device = torch.device('cpu')
        num_gpus = 0
    
    print(f"Device: {device}")
    print(f"GPUs: {num_gpus}")
    
    # Create small model for demo
    config = FSDPConfig(
        hidden_dim=256,
        num_heads=4,
        num_layers=4,
        ffn_dim=1024,
        batch_size=4,
    )
    
    model = OptimizedTransformer(config)
    model = model.to(device)
    
    # Use DataParallel if multiple GPUs
    if num_gpus > 1:
        model = nn.DataParallel(model)
    
    # Optimizer and scaler
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scaler = GradScaler(device=str(device))
    
    # Create dummy data
    batch_size = 4
    seq_length = 256
    input_ids = torch.randint(0, config.vocab_size, (batch_size, seq_length))
    labels = torch.randint(0, config.vocab_size, (batch_size, seq_length))
    
    # Training loop
    monitor = ResourceMonitor(device)
    num_steps = 5
    
    print(f"\nTraining for {num_steps} steps...")
    
    for step in range(num_steps):
        monitor.start_profiling()
        
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        
        # Mixed precision forward pass
        with autocast(device_type=str(device), dtype=torch.float16):
            logits = model(input_ids)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                labels.view(-1)
            )
        
        # Backward pass
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        
        elapsed, throughput = monitor.end_profiling(batch_size, seq_length)
        
        print(f"Step {step+1}/{num_steps}: loss={loss.item():.4f}, time={elapsed*1000:.1f}ms, throughput={throughput:.0f} tok/s")
    
    # Print summary
    monitor.print_summary()
    
    return model, monitor

# Run training example
model, monitor = run_optimized_training_example()

## 11. Summary: Multi-GPU Training Strategies

In [ ]:
def print_training_summary():
    """
    Summary of training strategies for different scales
    """
    
    summary = """
╔═══════════════════════════════════════════════════════════════════════════╗
║ MULTI-GPU TRAINING STRATEGIES SUMMARY                                    ║
╚═══════════════════════════════════════════════════════════════════════════╝

┌─ FOR SMALL MODELS (1-5B Parameters)
│
├─ Hardware: 1-4 GPUs (A100 80GB)
├─ Strategy: DataParallel or DDP
├─ Batch size: 32-64 per GPU
├─ Training time: Days to weeks
├─ Expected speedup: 3-4x on 4 GPUs
│
└─ Code:
   model = nn.DataParallel(model)  # Simple multi-GPU
   optimizer = torch.optim.AdamW(model.parameters())
   for batch in dataloader:
       output = model(batch)
       loss = criterion(output, target)
       loss.backward()
       optimizer.step()

┌─ FOR LARGE MODELS (10-70B Parameters)
│
├─ Hardware: 8-16 GPUs (A100 80GB)
├─ Strategy: Distributed Data Parallel (DDP)
├─ Batch size: 8-16 per GPU (total: 64-256)
├─ Training time: Weeks to months
├─ Expected speedup: 6-14x on 8-16 GPUs
│
└─ Launch command:
   torchrun --nproc_per_node=8 train.py --use-ddp

┌─ FOR MASSIVE MODELS (100B-1T Parameters)
│
├─ Hardware: 64-1024 GPUs (A100/H100)
├─ Strategy: FSDP + Tensor Parallelism + Pipeline Parallelism
├─ Batch size: 1-4 per GPU (accumulated: 128-4096)
├─ Training time: Months to years
├─ Expected speedup: 50-200x on 64-256 GPUs
│
└─ Configuration:
   - Use FSDP for memory sharding
   - Use Tensor Parallelism for layer distribution
   - Use Pipeline Parallelism for depth distribution
   - Overlap computation and communication

┌─ FOR M3 MACBOOK (Development & Fine-tuning)
│
├─ Hardware: Single GPU (8-128 GB unified memory)
├─ Strategy: Single GPU optimization + gradient checkpointing
├─ Batch size: 4-16 (small due to unified memory)
├─ Training time: Hours for small models, days for LLaMA-7B
├─ Expected speedup: 2.5-3.5x with optimization
├─ Models: Up to 7-13B with aggressive optimization
│
└─ Critical settings:
   ✓ num_workers=0 (NO MULTIPROCESSING!)
   ✓ use_mixed_precision=True (FP16 critical)
   ✓ gradient_checkpointing=True
   ✓ device='mps' (not 'cpu')

╔═══════════════════════════════════════════════════════════════════════════╗
║ RECOMMENDED WORKFLOW                                                      ║
╠═══════════════════════════════════════════════════════════════════════════╣
│                                                                            │
│ 1. DEVELOP on M3 MacBook (30 min - 2 hours)                              │
│    - Prototype model architecture                                         │
│    - Test training loop                                                   │
│    - Verify convergence                                                   │
│                                                                            │
│ 2. SCALE to cloud GPU (using code from step 1!)                           │
│    - Launch on 8-16 GPUs using torchrun                                  │
│    - Expected speedup: 6-14x                                             │
│    - Training time: days instead of weeks                                │
│                                                                            │
│ 3. DEPLOY back to M3                                                      │
│    - Quantize model (8-bit, 4-bit)                                       │
│    - Or distill to smaller student model                                 │
│    - Use for inference on M3                                             │
│                                                                            │
╚═══════════════════════════════════════════════════════════════════════════╝

KEY OPTIMIZATIONS BY IMPACT (on 8x GPU setup):

1. Mixed Precision (AMP):        2.5x speedup ⭐⭐⭐
2. Gradient Checkpointing:        1.3x (saves memory)
3. FSDP (vs DDP):                 1.2x (better scaling)
4. Overlapped Comm/Compute:       1.1-1.2x
5. Tensor Parallelism:            1.1-1.3x

TOTAL REALISTIC SPEEDUP:
- Single GPU:        1.0x (baseline)
- 8x GPU + DDP:      6-8x (after communication overhead)
- 16x GPU + FSDP:    12-16x
- 64x GPU + Full opt: 50-80x

M3 MACBOOK SPECIFIC:
- Speedup vs baseline:  2.5-3.5x with all optimizations
- vs single NVIDIA A100: 1/12x (A100 is 12x faster)
- Training 1B tokens on 7B model: ~30 hours on M3 Max 128GB
- Best use case: Development, fine-tuning, inference
    """
    
    print(summary)

print_training_summary()

## Next Steps

1. **For M3 MacBook Development**:
   - Use Module 2 (AMP) for 2.5x speedup
   - Use Module 4 (Gradient Checkpointing) to fit larger models
   - Follow M3_SPECIFIC_GUIDE.md for detailed optimizations
   - Train small models (< 7B parameters)

2. **For Multi-GPU Cloud Training**:
   - Follow LARGE_SCALE_GUIDE.md for architecture decisions
   - Use `torchrun` to launch DDP training
   - Monitor with ResourceMonitor from this notebook
   - Scale from 8 GPUs → 16 → 64 GPUs

3. **For Very Large Models (100B+)**:
   - Use FSDP from this notebook
   - Combine with Tensor Parallelism
   - Consider external libraries: DeepSpeed, Megatron-LM, Colossalai
   - Monitor communication vs computation time

4. **Key Resources**:
   - Read: LARGE_SCALE_GUIDE.md (comprehensive scaling guide)
   - Read: M3_SPECIFIC_GUIDE.md (M3 MacBook optimization)
   - Code example: scripts/train.py with --use-distributed flag
   - Model: models/llm.py (production-ready implementation)